In [ ]:
"""
ssh_daemon.py
-------------
Persistent background process that:
  - Listens for incoming encrypted credentials over LAN or WAN
  - Validates and stores credentials in a local hash-chained ledger (mini blockchain)
  - Reconnects automatically on network loss
  - Communicates with remote_desktop.py Flask server via shared protocol

Designed to run as a systemd service.

Usage (direct):
    python3 ssh_daemon.py

Usage (systemd):
    See credential-daemon.service
"""

import os
import json
import time
import socket
import signal
import hashlib
import logging
import threading
from datetime import datetime, timezone
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken

# ── Config ────────────────────────────────────────────────────────────────

LISTEN_HOST    = "0.0.0.0"       # bind all interfaces (LAN + WAN)
LISTEN_PORT    = int(os.environ.get("DAEMON_PORT", 9922))
KEY_PATH       = Path(os.environ.get("FERNET_KEY_PATH",  "/etc/credential-daemon/fernet.key"))
LEDGER_PATH    = Path(os.environ.get("LEDGER_PATH",      "/var/lib/credential-daemon/ledger.json"))
LOG_PATH       = Path(os.environ.get("LOG_PATH",         "/var/log/credential-daemon/daemon.log"))
RECONNECT_WAIT = int(os.environ.get("RECONNECT_WAIT",    5))    # seconds between retries
MAX_CONN_QUEUE = int(os.environ.get("MAX_CONN_QUEUE",    5))    # socket backlog
RECV_TIMEOUT   = int(os.environ.get("RECV_TIMEOUT",      30))   # per-connection timeout (s)
MAX_MSG_BYTES  = int(os.environ.get("MAX_MSG_BYTES",     4096))  # max incoming message size

# ── Logging ───────────────────────────────────────────────────────────────

LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(),          # also print to stdout → journald
    ]
)
log = logging.getLogger("credential-daemon")


In [ ]:

# ── Fernet key ────────────────────────────────────────────────────────────

def load_or_create_key() -> Fernet:
    KEY_PATH.parent.mkdir(parents=True, exist_ok=True)
    if KEY_PATH.exists():
        key = KEY_PATH.read_bytes()
        log.info(f"Loaded Fernet key from {KEY_PATH}")
    else:
        key = Fernet.generate_key()
        KEY_PATH.write_bytes(key)
        KEY_PATH.chmod(0o600)
        log.info(f"Generated new Fernet key at {KEY_PATH}")
    return Fernet(key)


In [ ]:

# ── Message protocol ──────────────────────────────────────────────────────
#
#  Every message is a newline-terminated JSON object:
#  { "type": "credential", "key_hash": "<hex>", "encrypted": "<hex>" }
#  { "type": "ping" }
#  { "type": "verify" }   → daemon responds with chain validity

def handle_message(raw: bytes, ledger: Ledger, fernet: Fernet) -> dict:
    try:
        msg = json.loads(raw.decode().strip())
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {"status": "error", "reason": "invalid JSON"}

    msg_type = msg.get("type")

    if msg_type == "ping":
        return {"status": "ok", "type": "pong"}

    if msg_type == "verify":
        valid = ledger.is_valid()
        return {"status": "ok", "chain_valid": valid, "length": len(ledger.chain)}

    if msg_type == "credential":
        key_hash  = msg.get("key_hash", "")
        encrypted = msg.get("encrypted", "")

        if not key_hash or not encrypted:
            return {"status": "error", "reason": "missing key_hash or encrypted"}

        # Verify we can decrypt it (proves we share the Fernet key)
        try:
            enc_bytes = bytes.fromhex(encrypted)
            fernet.decrypt(enc_bytes)
        except (InvalidToken, ValueError):
            return {"status": "error", "reason": "decryption failed — key mismatch or tampered payload"}

        block = ledger.append(key_hash, enc_bytes)
        return {"status": "ok", "block_index": block["index"], "block_hash": block["block_hash"]}

    return {"status": "error", "reason": f"unknown message type: {msg_type}"}


In [ ]:
# ── Connection handler ────────────────────────────────────────────────────

def handle_connection(conn: socket.socket, addr: tuple, ledger: Ledger, fernet: Fernet):
    ip, port = addr
    log.info(f"Connection from {ip}:{port}")
    conn.settimeout(RECV_TIMEOUT)
    try:
        data = conn.recv(MAX_MSG_BYTES)
        if not data:
            return
        response = handle_message(data, ledger, fernet)
        conn.sendall((json.dumps(response) + "\n").encode())
    except socket.timeout:
        log.warning(f"Connection from {ip} timed out.")
    except Exception as e:
        log.error(f"Error handling {ip}: {e}")
    finally:
        conn.close()


In [ ]:

# ── Daemon main loop ──────────────────────────────────────────────────────

class CredentialDaemon:
    def __init__(self):
        self.fernet  = load_or_create_key()
        self.ledger  = Ledger(LEDGER_PATH)
        self._stop   = threading.Event()

        signal.signal(signal.SIGTERM, self._handle_signal)
        signal.signal(signal.SIGINT,  self._handle_signal)

    def _handle_signal(self, signum, frame):
        log.info(f"Signal {signum} received — shutting down.")
        self._stop.set()

    def run(self):
        log.info(f"Credential daemon starting on {LISTEN_HOST}:{LISTEN_PORT}")
        while not self._stop.is_set():
            try:
                self._listen()
            except OSError as e:
                log.error(f"Socket error: {e} — retrying in {RECONNECT_WAIT}s")
                time.sleep(RECONNECT_WAIT)
        log.info("Daemon stopped.")

    def _listen(self):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as srv:
            srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            srv.bind((LISTEN_HOST, LISTEN_PORT))
            srv.listen(MAX_CONN_QUEUE)
            srv.settimeout(2.0)   # allows stop event to be checked
            log.info(f"Listening on port {LISTEN_PORT} (LAN + WAN)")

            while not self._stop.is_set():
                try:
                    conn, addr = srv.accept()
                    t = threading.Thread(
                        target=handle_connection,
                        args=(conn, addr, self.ledger, self.fernet),
                        daemon=True,
                    )
                    t.start()
                except socket.timeout:
                    continue  # just re-check stop event




In [ ]:
if __name__ == "__main__":
    CredentialDaemon().run()